# example 1

In [43]:
import time
from joblib import Parallel, delayed

def slow_square(x):
    time.sleep(1)  
    return x * x

def slow_square_series(inputs):
    results = [slow_square(x) for x in inputs]
    return results

def slow_square_parallel(inputs, njobs=-1, backend='loky'):
    results = Parallel(n_jobs=njobs, backend=backend)(
        delayed(slow_square)(x) for x in inputs
    )
    return results

In [ ]:
from joblib import cpu_count

inputs = range(cpu_count() - 1)
len(inputs)

13

In [34]:
slow_square_series(inputs)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144]

In [35]:
slow_square_parallel(inputs)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144]

In [39]:
import timeit

slow_square_series_time = timeit.timeit(
    lambda: slow_square_series(inputs),
    number=5
)
print(f'slow_square_series: {slow_square_series_time:.1f}')

slow_square_series: 65.2


In [46]:
for backend in ['loky', 'threading']:
    for njobs in [2, 4, -1]:
        print()
        print(f'backend={backend}, njobs={njobs}')
        slow_square_parallel_time = timeit.timeit(
            lambda: slow_square_parallel(inputs, njobs=njobs, backend=backend),
            number=5
        )
        print(f'slow_square_parallel: {slow_square_parallel_time:.1f}')
        print(f'{slow_square_series_time / slow_square_parallel_time:.1f} times speedup')


backend=loky, njobs=2
slow_square_parallel: 35.3
1.8 times speedup

backend=loky, njobs=4
slow_square_parallel: 20.2
3.2 times speedup

backend=loky, njobs=-1
slow_square_parallel: 5.3
12.3 times speedup

backend=threading, njobs=2
slow_square_parallel: 35.2
1.9 times speedup

backend=threading, njobs=4
slow_square_parallel: 20.1
3.2 times speedup

backend=threading, njobs=-1
slow_square_parallel: 5.1
12.8 times speedup


# example 2

In [47]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "group": np.random.choice(["A", "B", "C"], size=100),
    "value": np.random.randn(100)
})
print(df.shape)
df.head()

(100, 2)


,group,value
0,A,-0.904854
1,C,0.423389
2,B,0.798348
3,A,-0.739972
4,B,0.102363


In [49]:
grouped_df = df.groupby("group")

for name, group in grouped_df:
    print(name)

A
B
C


In [54]:
def process_group(name, group):
    result = group["value"].mean()
    return pd.DataFrame({
        "group": [name],
        "mean": [result]
    })

In [55]:
results = Parallel(n_jobs=-1)(
    delayed(process_group)(name, group)
    for name, group in grouped_df
)
results

[  group     mean
 0     A -0.25835,
   group      mean
 0     B  0.309214,
   group      mean
 0     C -0.206509]

In [56]:
pd.concat(results, ignore_index=True)

,group,mean
0,A,-0.258350
1,B,0.309214
2,C,-0.206509
